# Kimodo on Kaggle Free GPU — via Cloudflare Tunnel

Runs the full **Kimodo interactive demo** (viser UI, single port) on a Kaggle GPU session
and exposes it through a **free Cloudflare quick-tunnel** (`*.trycloudflare.com`) — no account needed.

**Before running:** set the notebook's *Accelerator* to **GPU** (P100 / T4 x2).
The text encoder is pinned to CPU so a 16 GB VRAM card works fine for generation.

| Stage | What happens |
|---|---|
| 1 | GPU check |
| 2 | Clone the repo (defaults to your extended fork) |
| 3 | Install deps (~5–10 min; C++ post-processor builds via CMake) |
| 4 | Environment + optional HF token (only for the gated SMPL-X model) |
| 5 | Launch `python -m kimodo.demo` and wait until healthy |
| 6 | Start `cloudflared`, print the public trycloudflare URL |
| 7 | Keep-alive loop |

> The tunnel URL changes on every session restart. Kaggle clears `/kaggle/working`
> between separate runs unless you cache to Kaggle datasets (see Troubleshooting).

In [ ]:
#@title 0. Configuration { display-mode: "form" }
REPO_URL = "https://github.com/liladhiee-web/kimodo-extended.git"  #@param ["https://github.com/liladhiee-web/kimodo-extended.git", "https://github.com/nv-tlabs/kimodo.git"]
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/kimodo"
SERVER_PORT = 7860
# With 16 GB VRAM cards the 8B LLM2Vec text encoder lives on CPU (a few seconds more per prompt)
TEXT_ENCODER_DEVICE = "cpu"  # "cuda" only if your card has 24+ GB VRAM
HF_TOKEN = ""  # optional: ONLY needed for the gated Kimodo-SMPLX-RP-v1 model
print("config ok ->", REPO_URL)

## 1 · GPU check

In [ ]:
import subprocess
r = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())
assert "NVIDIA" in r.stdout, "Enable the GPU accelerator in this notebook settings, then rerun"

## 2 · Clone the repo

Defaults to **your extended fork** `liladhiee-web/kimodo-extended.git` so the
`retarget/` tools and your local fixes are included. Switch the `REPO_URL` dropdown
to `nv-tlabs/kimodo.git` if you want the clean upstream instead.

In [ ]:
import os, shutil, subprocess as sp
if os.path.isdir(f"{REPO_DIR}/.git"):
    r = sp.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], capture_output=True, text=True)
    print("repo present, pull:", (r.stdout or r.stderr).strip() or "up-to-date")
else:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    os.makedirs(WORK, exist_ok=True)
    r = sp.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone failed: " + r.stderr[-2000:])
    print("cloned into", REPO_DIR)
os.chdir(REPO_DIR)
r2 = sp.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
print("git HEAD:", r2.stdout.strip())

## 3 · Install dependencies (~5–10 min)

Editable install with the `[demo,soma]` extras (viser fork + SOMA-X body model).
If the C++ `MotionCorrection` extension fails to build, we retry with it disabled —
post-processing then falls back to the pure-Python path.

In [ ]:
import os, subprocess, sys
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake"], check=False)

cmd = [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[demo,soma]"]
r1 = subprocess.run(cmd, capture_output=True, text=True)
if r1.returncode != 0:
    print("full install failed; retrying WITHOUT the MotionCorrection C++ build...")
    print((r1.stdout + r1.stderr)[-4000:])
    os.environ["SKIP_MOTION_CORRECTION_IN_SETUP"] = "1"
    r2 = subprocess.run(cmd, capture_output=True, text=True)
    if r2.returncode != 0:
        print((r2.stdout + r2.stderr)[-8000:])
        raise RuntimeError("pip install failed")
    print("installed without MotionCorrection (pure-python postprocess)")
else:
    print("installed OK (incl. MotionCorrection C++)")
import torch
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

## 4 · Environment + optional HF token

In [ ]:
import os
os.environ.pop("HUGGING_FACE_HUB_TOKEN", None)
if HF_TOKEN:
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

os.environ["TEXT_ENCODER_DEVICE"] = TEXT_ENCODER_DEVICE
os.environ["SERVER_NAME"] = "0.0.0.0"
os.environ["SERVER_PORT"] = str(SERVER_PORT)
os.environ.setdefault("PYTHONUNBUFFERED", "1")

# Model + checkpoint cache is ephemeral unless attached as a Kaggle dataset
os.environ.setdefault("XDG_CACHE_HOME", "/root/.cache")
print("env set:", TEXT_ENCODER_DEVICE, "encoder, port", SERVER_PORT)

## 5 · Launch the Kimodo demo

First launch downloads checkpoints (~6–8 GB: Kimodo + LLM2Vec) — **allow 8–15 minutes**.
Healthy once the viser server answers on port 7860.

In [ ]:
import os, subprocess, sys, time, urllib.request

subprocess.run(["pkill", "-f", "kimodo.demo"], capture_output=True)  # fresh start on reruns
log_path = f"{REPO_DIR}/demo.log"
demo_log = open(log_path, 'w')
demo_proc = subprocess.Popen(
    [sys.executable, "-m", "kimodo.demo"],
    cwd=REPO_DIR,
    stdout=demo_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
print("demo PID:", demo_proc.pid)

def tail(path, n=18):
    try:
        with open(path) as f:
            return "\n".join(f.read().splitlines()[-n:])
    except OSError:
        return "(log empty)"

ok = False
for i in range(240):
    time.sleep(3)
    if demo_proc.poll() is not None:
        break
    try:
        urllib.request.urlopen(f"http://localhost:{SERVER_PORT}/", timeout=2)
        ok = True
        break
    except Exception:
        pass

print("\n" + "=" * 60)
if ok:
    print(f"DEMO HEALTHY on localhost:{SERVER_PORT}")
else:
    print("NOT healthy yet — log tail:")
print(tail(log_path))

## 6 · Cloudflare tunnel → your public URL

Free **quick tunnel** (no Cloudflare account). Prints `https://<random>.trycloudflare.com`.
Viser UI + websockets work over this link. **Rerun this cell** for a fresh URL after restarts.

In [ ]:
import os, re, subprocess, time

CF_BIN = f"{WORK}/cloudflared"
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
if not os.path.exists(CF_BIN):
    subprocess.run(
        [
            "wget", "-q", "-O", CF_BIN,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ],
        check=True,
    )
subprocess.run(["chmod", "+x", CF_BIN])

tunnel_log = open(f"{WORK}/tunnel.log", 'wb')
tunnel_proc = subprocess.Popen(
    [CF_BIN, 'tunnel', '--no-autoupdate', f'--url=http://localhost:{SERVER_PORT}'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)
print("tunnel PID:", tunnel_proc.pid)

public_url = None
for _ in range(40):
    time.sleep(2)
    try:
        text = open(f"{WORK}/tunnel.log", 'rb').read().decode('utf-8', errors='replace')
    except OSError:
        continue
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", text)
    if m:
        public_url = m.group(0)
        break

print("=" * 72)
print("  YOUR PUBLIC KIMODO URL:")
print(f"      {public_url}" if public_url else "      (not ready yet — rerun this cell)")
print("=" * 72)

## 7 · Keep-alive

Prints every minute. Keep the tab open — with no kernel activity Kaggle terminates interactive sessions.

In [ ]:
import time, urllib.request
try:
    beat = 0
    while True:
        time.sleep(60)
        beat += 1
        state = "demo: up"
        try:
            urllib.request.urlopen(f"http://localhost:{SERVER_PORT}/", timeout=5)
        except Exception:
            state = "demo: DOWN (rerun Steps 5 & 6)"
        print(f"[{beat:>4}] {state}")
except KeyboardInterrupt:
    print("keepalive stopped — rerun cells 4-7 to come back")

## 8 · (Optional) CLI generation smoke test

Uses the same GPU as the demo. NPZ/BVH land in `/kaggle/working` (visible in the **Output**
panel). Avoid running this while a heavy demo generation is in progress.

In [ ]:
import os, subprocess, sys

out_dir = f"{WORK}/kimodo_cli_output"
os.makedirs(out_dir, exist_ok=True)
r = subprocess.run(
    [
        sys.executable, "-m", "kimodo.scripts.generate",
        "a person walking forward briskly",
        "--model", "soma-rp",
        "--duration", "3",
        "--num_samples", "1",
        "--output", out_dir,
    ],
    cwd=REPO_DIR,
    env=os.environ.copy(),
    capture_output=True,
    text=True,
)
print(r.stdout[-3000:] if r.stdout else '', r.stderr[-3000:] if r.returncode else '')
if r.returncode == 0:
    print("saved outputs in", out_dir)

## Troubleshooting

| Symptom | Fix |
|---|---|
| No GPU in step 1 | Notebook gear icon → Accelerator → GPU (P100 or T4 x2), restart |
| RAM OOM while loading the 8B text encoder | Prefer T4 x2 / P100 sessions; keep `TEXT_ENCODER_DEVICE = 'cpu'` |
| Tunnel URL dies / 530 errors | Rerun Step 6 — URLs are per-process |
| Model downloads repeat every session | Cache `~/.cache/huggingface` into a Kaggle Dataset, attach it, set `HF_HOME` to that path + `LOCAL_CACHE=true` in Step 4 |
| pip resolution failures on pins | Restart kernel → rerun Step 3 fresh |
| Slow first generation | Normal: checkpoint download + first batch compile |
| Want a stable URL instead of random | Needs a Cloudflare account + domain: `cloudflared tunnel login`, `--name kimodo`; quick tunnels are the zero-setup path |